# Steam Games — Data Cleaning

## 1. Load Data

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')

with open("../data/raw/games.json") as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient="index")
df.index.name = "AppID"
df = df.reset_index()

## 2. Initial Exploration

In [2]:
# df.describe()
# df.info()
print(f"Shape: {df.shape}")
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nNull counts:\n{df.isnull().sum().sort_values(ascending=False).head(20)}")
print(f"\nFirst 3 rows:")
df.head(3)
(df.isnull().sum() / len(df) * 100).round(1).sort_values(ascending=False)

Shape: (122611, 43)

Column names:
['AppID', 'name', 'release_date', 'required_age', 'price', 'dlc_count', 'detailed_description', 'about_the_game', 'short_description', 'reviews', 'header_image', 'website', 'support_url', 'support_email', 'windows', 'mac', 'linux', 'metacritic_score', 'metacritic_url', 'achievements', 'recommendations', 'notes', 'supported_languages', 'full_audio_languages', 'packages', 'developers', 'publishers', 'categories', 'genres', 'screenshots', 'movies', 'user_score', 'score_rank', 'positive', 'negative', 'estimated_owners', 'average_playtime_forever', 'average_playtime_2weeks', 'median_playtime_forever', 'median_playtime_2weeks', 'discount', 'peak_ccu', 'tags']

Data types:
AppID                           str
name                            str
release_date                    str
required_age                  int64
price                       float64
dlc_count                     int64
detailed_description            str
about_the_game                  str
sh

AppID                       0.0
score_rank                  0.0
packages                    0.0
developers                  0.0
publishers                  0.0
categories                  0.0
genres                      0.0
screenshots                 0.0
movies                      0.0
user_score                  0.0
positive                    0.0
supported_languages         0.0
negative                    0.0
estimated_owners            0.0
average_playtime_forever    0.0
average_playtime_2weeks     0.0
median_playtime_forever     0.0
median_playtime_2weeks      0.0
discount                    0.0
peak_ccu                    0.0
full_audio_languages        0.0
notes                       0.0
name                        0.0
header_image                0.0
release_date                0.0
required_age                0.0
price                       0.0
dlc_count                   0.0
detailed_description        0.0
about_the_game              0.0
short_description           0.0
reviews 

In [17]:
print(ds["price"].describe())
print(ds["review_score"].describe())
print(ds["release_year"].value_counts())
print(ds["tags"].head(5))

count    122611.000000
mean          4.765091
std          12.531030
min           0.000000
25%           0.550000
50%           2.240000
75%           5.240000
max         999.980000
Name: price, dtype: float64
count    82949.000000
mean         0.758276
std          0.238691
min          0.000000
25%          0.649682
50%          0.818182
75%          0.944444
max          1.000000
Name: review_score, dtype: float64
release_year
2025    24973
2024    20031
2023    14596
2022    12284
2021    11067
2020     8804
2018     7461
2019     7242
2017     5920
2016     4141
2015     2515
2014     1523
2013      467
2012      322
2009      321
2011      260
2010      255
2008      158
2007       90
2026       84
2006       69
2005        7
2004        6
2001        4
2003        3
2000        2
1999        2
1997        2
2002        1
1998        1
Name: count, dtype: int64
0                                                   []
1    {'Adventure': 27, 'Visual Novel': 19, 'Anime':...
2    {'C

## 3. Drop Irrelevant Columns
### Decisions:
- Dropped Achievements: 100% null
- Dropped User score: 96.5% null
- Dropped Header image: 90.2% null
- Dropped Average playtime forever: 81.7% null
- Dropped Support url: 59.5% null
- Dropped Support email: 55.8% null
- Dropped Screenshots: 32% null
- Dropped Metacritic score: 87% of values are 0, only meaningful 
  for 13% of games — not representative enough
- Dropped Reviews: language list, redundant with Supported languages
- Dropped Metacritic url, Score rank, Website, About the game, 
  Header image, Notes, Movies: URLs, text blobs, or not analytically useful
- Dropped Full audio languages, Supported languages: needs heavy 
  engineering, doesn't answer our business questions
- Dropped Categories: describes platform features not genre, 
  covered by Genres and Tags
- Dropped Average playtime two weeks, Median playtime two weeks: 
  keeping Median playtime forever as single playtime metric

In [4]:
ds = df.copy()
ds = ds.drop(columns=[
    "notes", "movies", "screenshots", "achievements",
    "user_score", "header_image", "average_playtime_forever",
    "average_playtime_2weeks", "median_playtime_2weeks",
    "support_url", "support_email", "website", "reviews",
    "full_audio_languages", "supported_languages", "about_the_game",
    "metacritic_url", "score_rank", "categories", "metacritic_score",
    "detailed_description", "short_description", "packages", "discount"
])

In [5]:
print(f"Shape: {ds.shape}")
print(f"\nColumn names:\n{ds.columns.tolist()}")

Shape: (122611, 19)

Column names:
['AppID', 'name', 'release_date', 'required_age', 'price', 'dlc_count', 'windows', 'mac', 'linux', 'recommendations', 'developers', 'publishers', 'genres', 'positive', 'negative', 'estimated_owners', 'median_playtime_forever', 'peak_ccu', 'tags']


## 4. Engineer New Features

### 4a. Create platform_count from Windows, Mac, Linux

In [6]:
ds["platform_count"] = ds["windows"].astype(int) + ds["mac"].astype(int) + ds["linux"].astype(int)

### 4b. Drop Windows, Mac, Linux and DiscountDLC count

In [7]:
ds = ds.drop(columns=["windows", "mac", "linux"])

### 4c. Convert Estimated owners → numeric (owners_mid)
Note: Estimated owners are provided as ranges (e.g. "20000 - 50000").
We extract the midpoint as a rough estimate. This is an approximation —
actual owner counts may vary significantly within each range.

In [8]:
ds = ds.rename(columns={"estimated_owners": "owners_mid"})
ds["owners_mid"] = ds["owners_mid"].apply(
    lambda x: (int(x.split(" - ")[0].strip()) + int(x.split(" - ")[1].strip())) / 2
)

### 4e. Convert Release date → datetime

In [9]:
ds["release_date"] = pd.to_datetime(ds["release_date"])

In [10]:
print(ds["release_date"].dtype)
print(ds["release_date"].head(5))

datetime64[us]
0   2023-08-01
1   2016-07-29
2   2019-05-06
3   2024-10-31
4   2025-04-24
Name: release_date, dtype: datetime64[us]


### 4f. Extract release_year and release_month as separate columns

In [11]:
ds["release_year"] = ds["release_date"].dt.year
ds["release_month"] = ds["release_date"].dt.month

### 4g. review_score

In [12]:
ds["review_score"] = np.where((ds["positive"] + ds["negative"]) > 0, ds["positive"] / (ds["positive"] +ds["negative"]), np.nan)

### Final checking dataset

In [13]:
print(ds.shape)
print(ds.columns.tolist())

(122611, 20)
['AppID', 'name', 'release_date', 'required_age', 'price', 'dlc_count', 'recommendations', 'developers', 'publishers', 'genres', 'positive', 'negative', 'owners_mid', 'median_playtime_forever', 'peak_ccu', 'tags', 'platform_count', 'release_year', 'release_month', 'review_score']


### 5. Save Cleaned Data

In [14]:
ds.to_csv("../data/processed/games_cleaned.csv", index=False)

### Check nulls

In [15]:
ds.isnull().sum().sort_values(ascending=False)

review_score               39662
name                           0
release_month                  0
release_year                   0
platform_count                 0
tags                           0
peak_ccu                       0
median_playtime_forever        0
owners_mid                     0
negative                       0
AppID                          0
genres                         0
publishers                     0
developers                     0
recommendations                0
dlc_count                      0
price                          0
required_age                   0
release_date                   0
positive                       0
dtype: int64